# 04 — Match Duration & Game Health

## Business Question
Is the game duration distribution healthy, and do longer games produce different outcomes than shorter ones?

## Statistical Depth
- KDE + normality testing (Kolmogorov-Smirnov)
- Mann-Whitney U tests comparing short vs long game groups
- Duration trends over the season (is the meta shifting?)
- Comeback analysis: do shorter games reflect snowballing?

In [1]:
import sys
sys.path.insert(0, '../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu

from config import *
from data_loader import load_matches
from stats_utils import test_normality, test_duration_groups
from plot_utils import set_style, save_plot

set_style()
df = load_matches()
print(f"Matches loaded: {len(df):,}")

Matches loaded: 51,490


## 4.1 — Normality Testing

In [2]:
# KS normality test
norm_result = test_normality(df['game_duration_min'].values, label='Game Duration')
print("=== Normality Test Results ===")
for k, v in norm_result.items():
    print(f"  {k}: {v}")

print(f"\nConclusion: Distribution is {'NORMAL' if norm_result['is_normal'] else 'NOT NORMAL'}")
print("=> Non-parametric tests (Mann-Whitney U) should be used for group comparisons")

# Visual test
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram + KDE
axes[0].hist(df['game_duration_min'], bins=80, color=COLORS['blue'], edgecolor='white', alpha=0.8, density=True)
kde_x = np.linspace(df['game_duration_min'].min(), df['game_duration_min'].max(), 300)
kde = stats.gaussian_kde(df['game_duration_min'])
axes[0].plot(kde_x, kde(kde_x), color=COLORS['red'], linewidth=2.5, label='Empirical KDE')
norm_x = np.linspace(df['game_duration_min'].min(), df['game_duration_min'].max(), 300)
norm_y = stats.norm.pdf(norm_x, df['game_duration_min'].mean(), df['game_duration_min'].std())
axes[0].plot(norm_x, norm_y, color=COLORS['green'], linewidth=2, linestyle='--', label='Normal fit')
axes[0].set_xlabel('Game Duration (minutes)')
axes[0].set_ylabel('Density')
axes[0].set_title('Duration Distribution\nvs Normal Curve')
axes[0].legend()

# Q-Q plot
stats.probplot(df['game_duration_min'], dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot\n(deviation from normality visible in tails)')
axes[1].get_lines()[0].set(color=COLORS['blue'], markersize=2, alpha=0.3)
axes[1].get_lines()[1].set(color=COLORS['red'], linewidth=2)

# CDF
sorted_dur = np.sort(df['game_duration_min'])
cdf = np.arange(1, len(sorted_dur)+1) / len(sorted_dur)
axes[2].plot(sorted_dur, cdf, color=COLORS['blue'], linewidth=2, label='Empirical CDF')
norm_cdf = stats.norm.cdf(sorted_dur, df['game_duration_min'].mean(), df['game_duration_min'].std())
axes[2].plot(sorted_dur, norm_cdf, color=COLORS['red'], linewidth=2, linestyle='--', label='Normal CDF')
axes[2].set_xlabel('Game Duration (minutes)')
axes[2].set_ylabel('Cumulative Probability')
axes[2].set_title('Empirical vs Normal CDF')
axes[2].legend()

plt.suptitle('Game Duration Normality Analysis', fontsize=14, fontweight='bold')
save_plot('04a_duration_normality.png')
plt.show()

=== Normality Test Results ===
  label: Game Duration
  ks_stat: 0.0393
  p_value: 0.0
  is_normal: False
  n: 51490
  mean: 30.54
  std: 8.53
  skewness: -0.3084
  kurtosis: 1.4871

Conclusion: Distribution is NOT NORMAL
=> Non-parametric tests (Mann-Whitney U) should be used for group comparisons
  Saved -> plots/04a_duration_normality.png


## 4.2 — Duration Group Comparisons (Mann-Whitney U Tests)

In [3]:
# Compare short vs long games
short_games = df[df['game_duration_min'] < 25]['game_duration_min'].values
long_games  = df[df['game_duration_min'] >= 35]['game_duration_min'].values
mid_games   = df[(df['game_duration_min'] >= 25) & (df['game_duration_min'] < 35)]['game_duration_min'].values

result = test_duration_groups(short_games, long_games, 'Short (<25 min)', 'Long (35+ min)')
print("=== Mann-Whitney U Test: Short vs Long Games ===")
for k, v in result.items():
    print(f"  {k}: {v}")

print(f"\nN short games: {len(short_games):,}")
print(f"N mid games: {len(mid_games):,}")
print(f"N long games: {len(long_games):,}")

# Objective count differences between short and long games
print("\n=== Objective differences: Short vs Long games ===")
for col in ['t1_baronKills', 't2_baronKills', 't1_dragonKills', 't2_dragonKills']:
    short_val = df[df['game_duration_min'] < 25][col].mean()
    long_val  = df[df['game_duration_min'] >= 35][col].mean()
    print(f"  {col}: short={short_val:.2f}, long={long_val:.2f}")

=== Mann-Whitney U Test: Short vs Long Games ===
  label1: Short (<25 min)
  label2: Long (35+ min)
  n1: 11697
  n2: 14602
  mean1: 19.55
  mean2: 40.29
  median1: 21.6
  median2: 39.05
  u_stat: 0.0
  p_value: 0.0
  significant: True
  cohens_d: -3.8921
  effect_label: large

N short games: 11,697
N mid games: 25,191
N long games: 14,602

=== Objective differences: Short vs Long games ===
  t1_baronKills: short=0.07, long=0.67
  t2_baronKills: short=0.08, long=0.76
  t1_dragonKills: short=0.74, long=1.98
  t2_dragonKills: short=0.70, long=2.00


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Duration by bucket — volume and win rate
bucket_stats = df.groupby('duration_bucket', observed=True).agg(
    games=('t1_won', 'count'),
    t1_win_rate=('t1_won', 'mean'),
    avg_baron=('t1_baronKills', 'mean'),
    avg_dragon=('t1_dragonKills', 'mean')
).reset_index()
bucket_stats['t1_win_rate_pct'] = (bucket_stats['t1_win_rate'] * 100).round(1)

x = range(len(bucket_stats))
bars = axes[0].bar(x, bucket_stats['games'], color=COLORS['blue'], edgecolor='white', alpha=0.85)
ax2 = axes[0].twinx()
ax2.plot(x, bucket_stats['t1_win_rate_pct'], color=COLORS['red'], marker='o',
         linewidth=2, markersize=8, label='Team 1 Win Rate')
ax2.axhline(50, color=COLORS['gray'], linestyle='--', linewidth=1.5, alpha=0.6)
ax2.set_ylabel('Team 1 Win Rate (%)', color=COLORS['red'])
ax2.set_ylim(40, 60)
axes[0].set_xticks(x)
axes[0].set_xticklabels([str(b) for b in bucket_stats['duration_bucket']], rotation=15)
axes[0].set_xlabel('Game Duration')
axes[0].set_ylabel('Number of Games')
axes[0].set_title('Game Volume and Win Rate by Duration')

# Season trend: weekly median duration
weekly = df.groupby('match_week')['game_duration_min'].agg(['median', 'mean', 'count']).reset_index()
weekly = weekly[weekly['count'] >= 50]
axes[1].plot(range(len(weekly)), weekly['median'], color=COLORS['blue'],
             linewidth=2, label='Median Duration')
axes[1].fill_between(range(len(weekly)),
    weekly['median'] - weekly['median'].std(),
    weekly['median'] + weekly['median'].std(),
    alpha=0.2, color=COLORS['blue'])
axes[1].set_xlabel('Week (Season 9, Jun–Sep 2017)')
axes[1].set_ylabel('Game Duration (minutes)')
axes[1].set_title('Median Game Duration by Week\n(Is the meta shifting?)')
axes[1].set_xticks(range(0, len(weekly), 2))
axes[1].set_xticklabels([str(w)[:7] for w in weekly['match_week'].iloc[::2]], rotation=45)
axes[1].legend()

plt.suptitle('Match Duration Analysis', fontsize=14, fontweight='bold')
save_plot('04b_duration_trends.png')
plt.show()

  Saved -> plots/04b_duration_trends.png


## Summary

| Finding | Result | Implication |
|---|---|---|
| Distribution normality | NOT normal (KS test) | Use non-parametric tests |
| Short vs long games | Statistically different (Mann-Whitney U) | Game experience varies significantly by duration |
| Season trend | Check weekly chart | Meta drift visible |
| Win rate by duration | ~50% across all buckets | No systematic advantage for longer games |

**Game Health Assessment:** The distribution of durations (mostly 25-35 minutes) is reasonable for a competitive MOBA. The peak around 30 minutes suggests most games reach meaningful late-game states before ending, without being excessively long.